# Покрытие v2 и бонус за ссылки на испорченных презентациях (EXP-024)

Основной запуск GRPO (EXP-023) поднял награду в основном за счёт снятых штрафов, а покрытие тезисов не сдвинулось
(0,597 → 0,599 на validation с поиском): пропущенный тезис почти ничего не стоит — ему достаётся «лучший» слайд
по теме со сходством ~0,5, и один слайд может быть лучшим сразу для нескольких тезисов. Модель перестала ставить
`#SOURCE`: за чужую ссылку штраф, за верную — ничего.

Проверяем поверх принятой награды (`accepted`, EXP-016/018):
- **`cov_scale`** — сходство тезиса пересчитывается в clip((s − lo)/(hi − lo), 0, 1); lo и hi откалиброваны
  на размеченных слайдах тезисов 16 исходников: lo — медиана сходства с лучшим *другим* слайдом той же презентации
  (что останется, если слайд тезиса убрать), hi — нижний квартиль сходства со *своим* слайдом;
- **`cov_cap2`, `cov_cap1`** — один слайд раскрывает не больше 2 (1) тезисов (жадное сопоставление); тезису без
  слайда — 0;
- **`cov_scale_cap2`** — оба;
- **`cov_share`** — доля раскрытых тезисов: тезис раскрыт, если по шкале `cov_scale` он выше середины (1), иначе 0;
- **`src_any`, `src_facts`** — бонус 0,1 × доля слайдов с верной `#SOURCE`; у `src_facts` слайд засчитывается,
  только если хоть одно его число или имя есть в выдаче.

Пары — те же, что в EXP-014/016/018 (16 исходников) и новые исходники (4, независимая проверка), плюс «общие слова
по теме» (у слайда тезиса заголовок остаётся, содержимое заменено фразой без содержания — тезис не раскрыт) и пары
для ссылок («ссылки убраны», «ссылка на каждом слайде» — лазейка). Ограничение: исходников мало и темы похожи, разница внутри пар небольшая — проверка
отсекает поломки и лазейки, но не доказывает пользу; дополнительно — совпадение с разметкой пар с конфликтом
(EXP-013) и пересчёт настоящих ответов GRPO (какие тезисы новая награда считает нераскрытыми). Правило выбора — `decisions.md`, раздел 7,
решение 16.11.2026, записано до прогона. Результат — `runs/perturbation/<дата>/`.

In [1]:
import csv, glob, json, os, statistics, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi: карта не должна быть занята
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl import perturb_eval as pe
from presentation_rl.rewards import Lemmatizer, load_key_points
from presentation_rl.tasks import load_tasks

OUT = ROOT / "runs" / "perturbation" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "runs" / "perturbation" / "embeddings"  # кеш эмбеддингов общий для всех запусков
EMBEDDER, ALPHA = "BAAI/bge-m3", 0.2
CURRENT = pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, "accepted")  # принятая награда
COVERAGE = [pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, v) for v in pe.COVERAGE_VARIANTS]
SOURCES = [pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, v) for v in pe.SOURCE_VARIANTS]
CONFIGS = [CURRENT, *COVERAGE, *SOURCES]
RUNS = [ROOT / "runs" / "variance" / "20261020_0331", ROOT / "runs" / "reward_sources" / "20261027_2150"]  # σ
MISSING = {"drop_explanation", "empty_body", "title_only_thesis", "foreign_body", "keys_only", "hide_in_notes",
           "hide_in_plan", "hide_in_image", "overloaded_slide", "generic_body"}  # тезис не раскрыт
INVARIANCE = {"paraphrase", "reorder"}
print(OUT)

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-yt2yi8wh because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


/home/jovyan/Diplom/runs/perturbation/20261116_1620


In [2]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(rows)


points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json")}
SETS = {}
for name, folder in (("old", ROOT / "data" / "perturbation"), ("holdout", ROOT / "data" / "perturbation" / "holdout")):
    items, pairs = pe.load_variants(folder / "variants")
    annotations = json.loads((folder / "annotations.json").read_text(encoding="utf-8"))
    extra, source_pairs = pe.source_variants(items)
    generic, generic_pairs = pe.generic_variants(items, annotations)
    items.update(extra)
    items.update(generic)
    SETS[name] = {"items": items, "pairs": pairs + generic_pairs + source_pairs, "annotations": annotations}
    print(f"{name}: вариантов {len(items)}, пар {len(pairs)} + общие слова {len(generic_pairs)} + для ссылок "
          f"{len(source_pairs)}, исходников {len(annotations)}")
contexts = {key[0]: item.context for key, item in SETS["old"]["items"].items() if key[1] == "original"}
natural = pe.load_natural(RUNS, tasks, set(contexts))
SETS["old"]["items"].update(natural)
conflict_items, conflicts = pe.load_conflicts(ROOT / "data" / "perturbation" / "conflicts", contexts)
SETS["old"]["items"].update(conflict_items)
pe.combine_judges(conflicts, "author", "codex", ["repeat_vs_missing"], "judge")  # судья — как в EXP-016/018
print("ответов T-lite для σ:", len(natural), "| пар с конфликтом:", len(conflicts))
start = time.time()
for name, s in SETS.items():
    pe.build_all(s["items"])
    s["pairs"], dropped = pe.check_pairs(s["pairs"], s["items"])
    print(f"{name}: после предусловий пар {len(s['pairs'])}, отброшено {len(dropped)}")
print(f"сборка {time.time() - start:.0f} с")

old: вариантов 1153, пар 1099 + общие слова 44 + для ссылок 10, исходников 16
holdout: вариантов 267, пар 254 + общие слова 9 + для ссылок 4, исходников 4
ответов T-lite для σ: 256 | пар с конфликтом: 80
собрано 200 из 1569
собрано 400 из 1569
собрано 600 из 1569
собрано 1200 из 1569
собрано 1400 из 1569
old: после предусловий пар 1153, отброшено 0
собрано 200 из 267
holdout: после предусловий пар 265, отброшено 2
сборка 478 с


## Шкала покрытия v2

lo и hi — только по 16 исходникам основного набора; на новых исходниках — проверка, разделяет ли шкала «свой»
и «чужой» слайд (доли «своих» с покрытием 1 и «чужих» с покрытием 0).

In [3]:
embedder = pe.CachedEmbedder(EMBEDDER, CACHE)
lemmatizer = Lemmatizer()
weights = CURRENT.weights()
SCALE, cal = pe.calibrate_coverage(SETS["old"]["items"], SETS["old"]["annotations"], points, embedder, lemmatizer, weights)
lo, hi = SCALE
_, cal_hold = pe.calibrate_coverage(SETS["holdout"]["items"], SETS["holdout"]["annotations"], points, embedder,
                                    lemmatizer, weights)
for r in cal_hold:  # новые исходники — со шкалой основного набора
    r.update(lo=lo, hi=hi, own_scaled=min(1.0, max(0.0, (r["own"] - lo) / (hi - lo))),
             other_scaled=min(1.0, max(0.0, (r["other"] - lo) / (hi - lo))))
embedder.save()
write_csv(OUT / "coverage_scale.csv", [{"set": "old", **r} for r in cal] + [{"set": "holdout", **r} for r in cal_hold])
print(f"шкала: lo = {lo:.4f}, hi = {hi:.4f}")
for name, rows in (("основной", cal), ("новые", cal_hold)):
    own, other = [r["own"] for r in rows], [r["other"] for r in rows]
    auc = statistics.mean(float(o > t) for o in own for t in other)
    print(f"  {name}: тезисов {len(rows)}; свой {statistics.median(own):.3f}, чужой {statistics.median(other):.3f}; "
          f"свой > чужой (по всем парам) {auc:.2f}; свой = 1: {statistics.mean(r['own_scaled'] >= 1 for r in rows):.2f}; "
          f"чужой = 0: {statistics.mean(r['other_scaled'] <= 0 for r in rows):.2f}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

шкала: lo = 0.5178, hi = 0.6505
  основной: тезисов 110; свой 0.719, чужой 0.518; свой > чужой (по всем парам) 0.93; свой = 1: 0.75; чужой = 0: 0.50
  новые: тезисов 19; свой 0.699, чужой 0.452; свой > чужой (по всем парам) 0.98; свой = 1: 0.79; чужой = 0: 0.74


In [4]:
scores, sigmas, hold_scores = {}, {}, {}
for config in CONFIGS:
    scores[config] = pe.score_items(SETS["old"]["items"], points, config, embedder, lemmatizer, scale=SCALE)
    sigmas[config], _ = pe.sigma(scores[config])
    hold_scores[config] = pe.score_items(SETS["holdout"]["items"], points, config, embedder, lemmatizer, scale=SCALE)
    embedder.save()
    print(f"{config.variant:15s} σ = {sigmas[config]:.4f}")
write_csv(OUT / "components.csv", ({"config": c.variant, "set": name, "key": "/".join(map(str, k)), **v}
                                   for name, sc in (("old", scores), ("holdout", hold_scores))
                                   for c, s in sc.items() for k, v in s.items()))

accepted        σ = 0.0829
cov_scale       σ = 0.1357
cov_cap2        σ = 0.0956
cov_scale_cap2  σ = 0.1374
cov_share       σ = 0.1717
cov_share_cap2  σ = 0.1685
cov_cap1        σ = 0.1077
src_any         σ = 0.0834
src_facts       σ = 0.0821


## Сдвиги по видам искажений

Медиана Δ награды в σ своей конфигурации и доля выполненных ожиданий (пороги от своей σ). Главное: виды «тезис
не раскрыт» (`MISSING`) должны стать заметно сильнее, перефразировка и перестановка — не хуже, остальные семейства —
без провалов.

In [5]:
def pair_rows(name, config):
    s = scores[config] if name == "old" else hold_scores[config]
    m, eps = pe.M_SHARE * sigmas[config], pe.EPS_SHARE * sigmas[config]  # σ — по ответам T-lite основного набора
    rows = pe.evaluate_pairs(SETS[name]["pairs"], s, m, eps)
    for r in rows:
        r["d_sigma"] = r["d_total"] / sigmas[config]
    return rows


ROWS = {(name, c): pair_rows(name, c) for name in SETS for c in CONFIGS}
write_csv(OUT / "pairs.csv", ({"set": name, "config": c.variant, **r} for (name, c), rows in ROWS.items() for r in rows))


def metrics(name, config):
    rows = ROWS[(name, config)]
    missing = [r["d_sigma"] for r in rows if r["kind"] in MISSING]
    invariance = [r["total"] == "ok" for r in rows if r["kind"] in INVARIANCE and r["total"] is not None]
    keys_only = [r["total"] in ("ok", "weak") for r in rows if r["kind"] == "keys_only"]
    shares = pe.family_shares(pe.tally([r for r in rows if r["family"] != "sources"]),
                              {t: 1 for t in {r["task"] for r in rows}})
    removed = [r["total"] in ("ok", "weak") for r in rows if r["kind"] == "sources_removed"]
    everywhere = [r["total"] == "ok" for r in rows if r["kind"] == "sources_everywhere"]
    mean = lambda v: statistics.mean(v) if v else None
    return {"missing": statistics.median(missing) if missing else None, "invariance": mean(invariance),
            "keys_only": mean(keys_only), "families": shares, "sources_removed": mean(removed),
            "sources_everywhere": mean(everywhere), "n_removed": len(removed), "n_everywhere": len(everywhere)}


METRICS = {(name, c): metrics(name, c) for name in SETS for c in CONFIGS}
fmt = lambda v: "—" if v is None else f"{v:+.2f}" if isinstance(v, float) and v < 0 else f"{v:.2f}"
for name in SETS:
    print(f"== {name}: медиана Δ «тезис не раскрыт» (σ) | перефразировка+перестановка ok | keys_only ok | "
          f"ссылки убраны ↓ | ссылка везде не ↑")
    for c in CONFIGS:
        m = METRICS[(name, c)]
        print(f"  {c.variant:15s} {fmt(m['missing']):>6s} | {fmt(m['invariance']):>5s} | {fmt(m['keys_only']):>5s} | "
              f"{fmt(m['sources_removed'])} (n={m['n_removed']}) | {fmt(m['sources_everywhere'])} (n={m['n_everywhere']})")
    if name == "old":
        for c in CONFIGS:
            rows = pe.conflict_rows(conflicts, scores[c], "judge", pe.EPS_SHARE * sigmas[c])
            if rows:
                print(f"  пары с конфликтом, совпадение с судьёй: {c.variant:15s} "
                      f"{statistics.mean(r['agree'] for r in rows):.2f} (n={len(rows)})")
    print("  семейства:", "; ".join(f"{c.variant}: " + ", ".join(f"{f} {v:.2f}" for f, v in sorted(METRICS[(name, c)]['families'].items()))
                                   for c in CONFIGS))
summary = []
for name in SETS:
    for c in CONFIGS:
        for e in pe.summarize(ROWS[(name, c)]):
            summary.append({"set": name, "config": c.variant, "sigma": sigmas[c], **e,
                            "d_total_median_sigma": e["d_total_median"] / sigmas[c]})
write_csv(OUT / "summary.csv", summary)

== old: медиана Δ «тезис не раскрыт» (σ) | перефразировка+перестановка ok | keys_only ok | ссылки убраны ↓ | ссылка везде не ↑
  accepted         -0.69 |  0.67 |  0.09 | 0.00 (n=3) | 1.00 (n=7)
  cov_scale        -0.76 |  0.84 |  0.30 | 0.00 (n=3) | 1.00 (n=7)
  cov_cap2         -0.60 |  0.70 |  0.07 | 0.00 (n=3) | 1.00 (n=7)
  cov_scale_cap2   -0.75 |  0.83 |  0.30 | 0.00 (n=3) | 1.00 (n=7)
  cov_share        -0.77 |  0.92 |  0.23 | 0.00 (n=3) | 1.00 (n=7)
  cov_share_cap2   -0.79 |  0.89 |  0.23 | 0.00 (n=3) | 1.00 (n=7)
  cov_cap1         -0.56 |  0.71 |  0.07 | 0.00 (n=3) | 1.00 (n=7)
  src_any          -0.69 |  0.67 |  0.09 | 1.00 (n=3) | 0.14 (n=7)
  src_facts        -0.70 |  0.67 |  0.09 | 1.00 (n=3) | 0.14 (n=7)
  пары с конфликтом, совпадение с судьёй: accepted        0.73 (n=60)
  пары с конфликтом, совпадение с судьёй: cov_scale       0.62 (n=60)
  пары с конфликтом, совпадение с судьёй: cov_cap2        0.73 (n=60)
  пары с конфликтом, совпадение с судьёй: cov_scale_cap2  0.

## Выбор по правилу (решение 16.11.2026, до прогона)

**Покрытие.** Вариант допустим, если на основном наборе **и** на новых исходниках: (а) медиана Δ «тезис не раскрыт»
ниже, чем у принятой, хотя бы на 0,25σ; (б) доля выполненных ожиданий перефразировки и перестановки не ниже, чем
у принятой, минус 0,10; (в) `keys_only` выполняется не реже, чем у принятой; (г) доли остальных семейств (кроме
покрытия и инвариантности) не ниже, чем у принятой, минус 0,05. Из допустимых — с наибольшим усилением на новых
исходниках; при разнице меньше 0,1σ — более простой по порядку `cov_scale`, `cov_cap2`, `cov_scale_cap2`,
`cov_share`, `cov_cap1`.
Нет допустимых — покрытие прежнее.

**Ссылки.** Вариант допустим, если по обоим наборам вместе: «ссылки убраны» снижают награду (выполнено или слабо)
не меньше чем в 70% пар, «ссылка на каждом слайде» не повышает её не меньше чем в 80% пар, а доли остальных
семейств на основном наборе не ниже, чем у принятой, минус 0,05. Из допустимых — `src_facts` (строже), иначе
`src_any`; нет допустимых — бонуса нет.

In [6]:
def other_families_ok(name, config, tolerance=0.05):
    base, mine = METRICS[(name, CURRENT)]["families"], METRICS[(name, config)]["families"]
    return all(mine.get(f, 0) >= v - tolerance for f, v in base.items() if f not in ("coverage", "invariance"))


coverage_report = []
for c in COVERAGE:
    checks = {}
    for name in SETS:
        m, b = METRICS[(name, c)], METRICS[(name, CURRENT)]
        checks[name] = {
            "missing": m["missing"] is not None and m["missing"] <= b["missing"] - 0.25,
            "invariance": m["invariance"] is None or m["invariance"] >= b["invariance"] - 0.10,
            "keys_only": m["keys_only"] is None or b["keys_only"] is None or m["keys_only"] >= b["keys_only"],
            "families": other_families_ok(name, c)}
    ok = all(all(v.values()) for v in checks.values())
    gain = METRICS[("holdout", CURRENT)]["missing"] - METRICS[("holdout", c)]["missing"]
    coverage_report.append({"variant": c.variant, "admissible": ok, "gain_holdout": gain, **{
        f"{name}_{k}": v for name, part in checks.items() for k, v in part.items()}})
    print(c.variant, "допустим" if ok else "нет", f"усиление на новых {gain:+.2f}σ", checks)
order = {v: i for i, v in enumerate(pe.COVERAGE_VARIANTS)}
admissible = sorted((r for r in coverage_report if r["admissible"]), key=lambda r: -r["gain_holdout"])
if admissible:
    best = admissible[0]["gain_holdout"]
    close = [r for r in admissible if best - r["gain_holdout"] < 0.1]
    CHOSEN_COVERAGE = min(close, key=lambda r: order[r["variant"]])["variant"]
else:
    CHOSEN_COVERAGE = None

source_report = []
for c in SOURCES:
    removed, everywhere = [], []
    for name in SETS:
        rows = ROWS[(name, c)]
        removed += [r["total"] in ("ok", "weak") for r in rows if r["kind"] == "sources_removed"]
        everywhere += [r["total"] == "ok" for r in rows if r["kind"] == "sources_everywhere"]
    ok = bool(removed and everywhere and statistics.mean(removed) >= 0.7 and statistics.mean(everywhere) >= 0.8
              and other_families_ok("old", c))
    source_report.append({"variant": c.variant, "admissible": ok, "removed_down": statistics.mean(removed) if removed else None,
                          "everywhere_not_up": statistics.mean(everywhere) if everywhere else None,
                          "n_removed": len(removed), "n_everywhere": len(everywhere)})
    print(source_report[-1])
CHOSEN_SOURCES = next((r["variant"] for v in ("src_facts", "src_any") for r in source_report
                       if r["variant"] == v and r["admissible"]), None)

write_csv(OUT / "choice.csv", coverage_report + source_report)
(OUT / "choice.json").write_text(json.dumps({"coverage": CHOSEN_COVERAGE, "sources": CHOSEN_SOURCES, "scale": SCALE},
                                            ensure_ascii=False, indent=1), encoding="utf-8")
print("ИТОГ: покрытие —", CHOSEN_COVERAGE or "прежнее", "| ссылки —", CHOSEN_SOURCES or "без бонуса", "| шкала", SCALE)

cov_scale нет усиление на новых -0.01σ {'old': {'missing': False, 'invariance': True, 'keys_only': True, 'families': False}, 'holdout': {'missing': False, 'invariance': False, 'keys_only': True, 'families': False}}
cov_cap2 нет усиление на новых -0.16σ {'old': {'missing': False, 'invariance': True, 'keys_only': False, 'families': False}, 'holdout': {'missing': False, 'invariance': True, 'keys_only': True, 'families': False}}
cov_scale_cap2 нет усиление на новых -0.03σ {'old': {'missing': False, 'invariance': True, 'keys_only': True, 'families': False}, 'holdout': {'missing': False, 'invariance': False, 'keys_only': True, 'families': False}}
cov_share нет усиление на новых -0.04σ {'old': {'missing': False, 'invariance': True, 'keys_only': True, 'families': False}, 'holdout': {'missing': False, 'invariance': True, 'keys_only': False, 'families': False}}
cov_share_cap2 нет усиление на новых -0.01σ {'old': {'missing': False, 'invariance': True, 'keys_only': True, 'families': False}, 'holdo

## Для разбора: ответы GRPO по-новому (не участвует в выборе)

Ответы исходной модели и шага 130 основного запуска GRPO на 8 темах validation — принятая награда и выбранный
вариант: сколько тезисов остаются нераскрытыми и как меняется покрытие.

In [7]:
RUN = sorted((ROOT / "runs" / "grpo").glob("2*_seed0"))[-1]
FOLDERS = {"исходная": ROOT / "runs" / "grpo" / "reference" / "episodes", "шаг 130": RUN / "checkpoints" / "step_0130" / "validation"}
topics = json.loads((RUN / "manifest.json").read_text(encoding="utf-8"))["validation_topics"]
from presentation_rl.perturb import task_context

grpo_items = {}
for label, folder in FOLDERS.items():
    for meta in sorted(folder.glob("*_*_*.json")):
        r = json.loads(meta.read_text(encoding="utf-8"))
        if r["task_id"] not in topics:
            continue
        stem = meta.stem
        text = (folder / f"{stem}.txt").read_text(encoding="utf-8")
        key = (r["task_id"], label, stem)
        grpo_items[key] = pe.Item(key, r["task_id"], (folder / f"{stem}_answer.txt").read_text(encoding="utf-8"),
                                  task_context(tasks[r["task_id"]], text, r["mode"] == "search"))
pe.build_all(grpo_items)
shown = [CURRENT] + [c for c in CONFIGS if c.variant in (CHOSEN_COVERAGE, CHOSEN_SOURCES)]
for c in shown:
    s = pe.score_items(grpo_items, points, c, embedder, lemmatizer, scale=SCALE)
    for label in FOLDERS:
        for mode in ("search", "plain"):
            part = [v for k, v in s.items() if k[1] == label and k[2].startswith(mode)]
            print(f"{c.variant:15s} {label:9s} {mode:6s} награда {statistics.mean(v['total'] for v in part):.3f} "
                  f"покрытие {statistics.mean(v['coverage'] for v in part):.3f} "
                  f"нераскрытых тезисов {statistics.mean(v['uncovered'] for v in part):.2f} "
                  f"ссылки {statistics.mean(v['sources'] for v in part):.2f}")
embedder.save()
# какие тезисы новая награда считает нераскрытыми (сверить с разбором ответов вручную: t376, t389)
from presentation_rl.rewards import score as reward_score
if CHOSEN_COVERAGE:
    w = next(c for c in CONFIGS if c.variant == CHOSEN_COVERAGE).weights(scale=SCALE)
    for key, item in sorted(grpo_items.items()):
        c = item.context
        r = reward_score(item.deck, points[item.task], conclusion=c["conclusion"],
                         slide_range=tuple(c["slide_range"]) if c["slide_range"] else None, embedder=embedder,
                         lemmatizer=lemmatizer, report=item.report, images_allowed=c["images_allowed"],
                         tool_text=c.get("tool_text"), calls=c.get("calls"), search=c["search"], weights=w)
        lo_, hi_ = SCALE
        missing = [p.title for p, t in zip(points[item.task], r.theses)
                   if t.best_slide is None or (t.similarity - lo_) / (hi_ - lo_) <= 0]
        print(f"{key[1]:9s} {key[2]:22s} нераскрыто {len(missing)}: {'; '.join(missing)[:200]}")

accepted        исходная  search награда 0.597 покрытие 0.597 нераскрытых тезисов 0.00 ссылки 0.00
accepted        исходная  plain  награда 0.734 покрытие 0.603 нераскрытых тезисов 0.00 ссылки 0.00
accepted        шаг 130   search награда 0.761 покрытие 0.593 нераскрытых тезисов 0.00 ссылки 0.00
accepted        шаг 130   plain  награда 0.813 покрытие 0.606 нераскрытых тезисов 0.00 ссылки 0.00
